# Mölnlycke Health Care Demo - Data Ingestion

This notebook generates the POC database for Mölnlycke competitive intelligence demo across the European wound care and surgical products market.

In [2]:
import os
from snowflake.snowpark import Session

session = Session.builder.config("connection_name", os.getenv("SNOWFLAKE_CONNECTION_NAME", "oregon_tp")).create()
print(f"Connected as: {session.get_current_user()}")
print(f"Role: {session.get_current_role()}")
print(f"Warehouse: {session.get_current_warehouse()}")

Connected as: "admin"
Role: "ACCOUNTADMIN"
Warehouse: "AI_WH"


## Step 1: Create Database and Schema

In [3]:
session.sql("CREATE OR REPLACE DATABASE MOLNLYCKE_DEMO").collect()
session.sql("CREATE OR REPLACE SCHEMA MOLNLYCKE_DEMO.ANALYTICS").collect()
session.sql("USE SCHEMA MOLNLYCKE_DEMO.ANALYTICS").collect()
print("Database and schema created.")

Database and schema created.


## Step 2: Create Tables

In [4]:
session.sql("""
CREATE OR REPLACE TABLE DIM_PRODUCTS (
    PRODUCT_ID INT PRIMARY KEY,
    PRODUCT_NAME VARCHAR(200) NOT NULL,
    MANUFACTURER VARCHAR(100),
    HEADQUARTERS VARCHAR(100),
    BUSINESS_AREA VARCHAR(50),
    PRODUCT_TYPE VARCHAR(50),
    PRICE_TIER VARCHAR(20),
    IS_MOLNLYCKE_PRODUCT BOOLEAN DEFAULT FALSE,
    PRIMARY_CATEGORY VARCHAR(100),
    EU_HOSPITAL_COUNT INT
)
""").collect()

session.sql("""
CREATE OR REPLACE TABLE DIM_PRODUCT_CATEGORIES (
    CATEGORY_ID INT PRIMARY KEY,
    CATEGORY_NAME VARCHAR(100) NOT NULL,
    SEGMENT VARCHAR(50),
    AVG_PRICE_EUR INT,
    GROWTH_TREND VARCHAR(20)
)
""").collect()

session.sql("""
CREATE OR REPLACE TABLE BRIDGE_PRODUCT_CATEGORIES (
    PRODUCT_ID INT,
    CATEGORY_ID INT,
    PRIMARY KEY (PRODUCT_ID, CATEGORY_ID)
)
""").collect()

session.sql("""
CREATE OR REPLACE TABLE FACT_PRODUCT_METRICS (
    PRODUCT_ID INT,
    YEAR_QUARTER DATE,
    EU_UNITS_SOLD INT,
    EU_REVENUE_EUR INT,
    AVG_SELLING_PRICE_EUR INT,
    EU_MARKET_SHARE_PCT FLOAT,
    CLINICAL_SATISFACTION_SCORE FLOAT,
    NEW_SKU_LAUNCHES INT,
    PRIMARY KEY (PRODUCT_ID, YEAR_QUARTER)
)
""").collect()

session.sql("""
CREATE OR REPLACE TABLE REVIEWS (
    REVIEW_ID INT PRIMARY KEY,
    PRODUCT_ID INT NOT NULL,
    PRODUCT_CATEGORY VARCHAR(100),
    REVIEWER_NAME VARCHAR(100),
    REVIEWER_COUNTRY VARCHAR(100),
    REVIEWER_ROLE VARCHAR(100),
    RATING INT,
    REVIEW_DATE DATE,
    REVIEW_TITLE VARCHAR(500),
    REVIEW_TEXT VARCHAR(4000),
    CARE_SETTING VARCHAR(50),
    WOUND_TYPE VARCHAR(50)
)
""").collect()

print("All tables created successfully.")

All tables created successfully.


## Step 3: Generate Product Data

In [5]:
import pandas as pd

products_data = [
    (1, "Mepilex", "Mölnlycke", "Sweden", "Wound Care", "Foam Dressings", "Premium", True, "Foam Dressings", 12000),
    (2, "Mepitel", "Mölnlycke", "Sweden", "Wound Care", "Wound Contact Layers", "Premium", True, "Wound Contact Layers", 8000),
    (3, "Exufiber", "Mölnlycke", "Sweden", "Wound Care", "Gelling Fiber", "Premium", True, "Alginate/Fiber Dressings", 6000),
    (4, "Melgisorb", "Mölnlycke", "Sweden", "Wound Care", "Antimicrobial", "Premium", True, "Antimicrobial Dressings", 5500),
    (5, "Avance Solo", "Mölnlycke", "Sweden", "Wound Care", "NPWT", "Premium", True, "NPWT", 4000),
    (6, "Granudacyn", "Mölnlycke", "Sweden", "Wound Care", "Wound Cleansing", "Mid-Range", True, "Wound Cleansing", 7000),
    (7, "Biogel", "Mölnlycke", "Sweden", "Gloves", "Surgical Gloves", "Premium", True, "Surgical Gloves", 10000),
    (8, "BARRIER", "Mölnlycke", "Sweden", "OR Solutions", "Drapes/Gowns", "Mid-Range", True, "Surgical Drapes/Gowns", 9000),
    (9, "ProcedurePak", "Mölnlycke", "Sweden", "OR Solutions", "Procedure Trays", "Premium", True, "Surgical Drapes/Gowns", 6500),
    (10, "Hibiclens", "Mölnlycke", "Sweden", "Antiseptics", "Antiseptic Solution", "Mid-Range", True, "Antiseptic Solutions", 8500),

    (11, "ALLEVYN", "Smith+Nephew", "UK", "Wound Care", "Foam Dressings", "Premium", False, "Foam Dressings", 14000),
    (12, "PICO", "Smith+Nephew", "UK", "Wound Care", "NPWT", "Premium", False, "NPWT", 8000),
    (13, "ACTICOAT", "Smith+Nephew", "UK", "Wound Care", "Antimicrobial", "Premium", False, "Antimicrobial Dressings", 6000),
    (14, "Aquacel", "ConvaTec", "UK", "Wound Care", "Hydrofiber", "Premium", False, "Alginate/Fiber Dressings", 11000),
    (15, "Aquacel Foam", "ConvaTec", "UK", "Wound Care", "Foam Dressings", "Premium", False, "Foam Dressings", 9000),
    (16, "Aquacel Ag+", "ConvaTec", "UK", "Wound Care", "Antimicrobial", "Premium", False, "Antimicrobial Dressings", 7500),
    (17, "Biatain", "Coloplast", "Denmark", "Wound Care", "Foam Dressings", "Premium", False, "Foam Dressings", 10000),
    (18, "Biatain Ag", "Coloplast", "Denmark", "Wound Care", "Antimicrobial", "Mid-Range", False, "Antimicrobial Dressings", 6500),
    (19, "Tegaderm", "Solventum", "USA", "Wound Care", "Film Dressings", "Mid-Range", False, "Foam Dressings", 13000),
    (20, "V.A.C.", "Solventum", "USA", "Wound Care", "NPWT", "Premium", False, "NPWT", 9500),
    (21, "Askina", "B. Braun", "Germany", "Wound Care", "Foam/Alginate", "Mid-Range", False, "Foam Dressings", 8000),
    (22, "Prontosan", "B. Braun", "Germany", "Wound Care", "Wound Cleansing", "Mid-Range", False, "Wound Cleansing", 7000),
    (23, "Cutimed", "Essity", "Sweden", "Wound Care", "Foam Dressings", "Budget", False, "Foam Dressings", 7500),
    (24, "Jobst", "Essity", "Sweden", "Wound Care", "Compression", "Mid-Range", False, "Compression Therapy", 6000),
    (25, "Zetuvit", "Paul Hartmann", "Germany", "Wound Care", "Superabsorbent", "Budget", False, "Superabsorbent Dressings", 5000),
]

pdf = pd.DataFrame(products_data, columns=[
    "PRODUCT_ID", "PRODUCT_NAME", "MANUFACTURER", "HEADQUARTERS", "BUSINESS_AREA",
    "PRODUCT_TYPE", "PRICE_TIER", "IS_MOLNLYCKE_PRODUCT", "PRIMARY_CATEGORY", "EU_HOSPITAL_COUNT"
])

df = session.create_dataframe(pdf)
df.write.mode("overwrite").save_as_table("DIM_PRODUCTS")
print(f"Inserted {session.table('DIM_PRODUCTS').count()} products")

Inserted 25 products


## Step 4: Generate Product Categories

In [6]:
categories_data = [
    (1, "Foam Dressings", "Wound Care", 15, "Stable"),
    (2, "Antimicrobial Dressings", "Wound Care", 25, "Growing"),
    (3, "Alginate/Fiber Dressings", "Wound Care", 18, "Growing"),
    (4, "Wound Contact Layers", "Wound Care", 12, "Stable"),
    (5, "NPWT", "Wound Care", 180, "Growing"),
    (6, "Superabsorbent Dressings", "Wound Care", 10, "Stable"),
    (7, "Wound Cleansing", "Wound Care", 8, "Growing"),
    (8, "Scar Management", "Wound Care", 20, "Stable"),
    (9, "Surgical Gloves", "Surgical", 3, "Stable"),
    (10, "Surgical Drapes/Gowns", "Surgical", 35, "Stable"),
    (11, "Antiseptic Solutions", "Infection Prevention", 6, "Growing"),
    (12, "Compression Therapy", "Wound Care", 22, "Growing"),
]

pdf_categories = pd.DataFrame(categories_data, columns=[
    "CATEGORY_ID", "CATEGORY_NAME", "SEGMENT", "AVG_PRICE_EUR", "GROWTH_TREND"
])
session.create_dataframe(pdf_categories).write.mode("overwrite").save_as_table("DIM_PRODUCT_CATEGORIES")
print(f"Inserted {session.table('DIM_PRODUCT_CATEGORIES').count()} product categories")

Inserted 12 product categories


## Step 5: Create Product-Category Mappings

In [7]:
product_categories = [
    (1, 1),                     # Mepilex -> Foam Dressings
    (2, 4),                     # Mepitel -> Wound Contact Layers
    (3, 3),                     # Exufiber -> Alginate/Fiber
    (4, 2),                     # Melgisorb -> Antimicrobial
    (5, 5),                     # Avance Solo -> NPWT
    (6, 7),                     # Granudacyn -> Wound Cleansing
    (7, 9),                     # Biogel -> Surgical Gloves
    (8, 10),                    # BARRIER -> Surgical Drapes/Gowns
    (9, 10),                    # ProcedurePak -> Surgical Drapes/Gowns
    (10, 11),                   # Hibiclens -> Antiseptic Solutions
    (11, 1), (11, 6),           # ALLEVYN -> Foam, Superabsorbent
    (12, 5),                    # PICO -> NPWT
    (13, 2),                    # ACTICOAT -> Antimicrobial
    (14, 3), (14, 1),           # Aquacel -> Alginate/Fiber, Foam
    (15, 1),                    # Aquacel Foam -> Foam
    (16, 2),                    # Aquacel Ag+ -> Antimicrobial
    (17, 1), (17, 6),           # Biatain -> Foam, Superabsorbent
    (18, 2),                    # Biatain Ag -> Antimicrobial
    (19, 1), (19, 4),           # Tegaderm -> Foam, Wound Contact Layers
    (20, 5),                    # V.A.C. -> NPWT
    (21, 1), (21, 3),           # Askina -> Foam, Alginate
    (22, 7),                    # Prontosan -> Wound Cleansing
    (23, 1), (23, 6),           # Cutimed -> Foam, Superabsorbent
    (24, 12),                   # Jobst -> Compression
    (25, 6),                    # Zetuvit -> Superabsorbent
]

pdf_pc = pd.DataFrame(product_categories, columns=["PRODUCT_ID", "CATEGORY_ID"])
session.create_dataframe(pdf_pc).write.mode("overwrite").save_as_table("BRIDGE_PRODUCT_CATEGORIES")
print(f"Created {len(product_categories)} product-category mappings")

Created 31 product-category mappings


## Step 6: Generate Quarterly Product Metrics

In [8]:
from datetime import datetime
from dateutil.relativedelta import relativedelta
import random

random.seed(42)

products_df = session.table("DIM_PRODUCTS").to_pandas()

base_quarterly_units = {
    "Mepilex": 180000, "Mepitel": 90000, "Exufiber": 60000, "Melgisorb": 45000,
    "Avance Solo": 15000, "Granudacyn": 70000, "Biogel": 200000, "BARRIER": 150000,
    "ProcedurePak": 80000, "Hibiclens": 120000,
    "ALLEVYN": 220000, "PICO": 25000, "ACTICOAT": 50000,
    "Aquacel": 190000, "Aquacel Foam": 140000, "Aquacel Ag+": 65000,
    "Biatain": 160000, "Biatain Ag": 55000,
    "Tegaderm": 250000, "V.A.C.": 30000,
    "Askina": 100000, "Prontosan": 60000,
    "Cutimed": 85000, "Jobst": 50000, "Zetuvit": 40000,
}

avg_price = {"Budget": 8, "Mid-Range": 15, "Premium": 28}

start_quarter = datetime(2023, 1, 1)
end_quarter = datetime(2025, 10, 1)

metrics_data = []
current_quarter = start_quarter

while current_quarter <= end_quarter:
    quarter_total_units = 0
    quarter_product_units = {}

    for _, product in products_df.iterrows():
        product_id = int(product["PRODUCT_ID"])
        product_name = product["PRODUCT_NAME"]
        price_tier = product["PRICE_TIER"]

        base = base_quarterly_units.get(product_name, 50000)

        quarter_num = (current_quarter.month - 1) // 3 + 1
        if quarter_num in [1, 4]:
            seasonality = 1.05
        else:
            seasonality = 0.97

        noise = random.uniform(0.88, 1.12)
        units = int(base * seasonality * noise)
        quarter_product_units[product_id] = units
        quarter_total_units += units

    for product_id, units in quarter_product_units.items():
        product_row = products_df[products_df["PRODUCT_ID"] == product_id].iloc[0]
        price_tier = product_row["PRICE_TIER"]

        asp = int(avg_price.get(price_tier, 15) * random.uniform(0.9, 1.1))
        revenue = units * asp
        market_share = round(units / quarter_total_units * 100, 2)
        satisfaction = round(random.uniform(3.6, 4.7), 2)
        new_skus = random.choices([0, 0, 0, 0, 0, 1, 1, 2], k=1)[0]

        metrics_data.append((
            product_id,
            current_quarter.strftime("%Y-%m-01"),
            units,
            revenue,
            asp,
            market_share,
            satisfaction,
            new_skus
        ))

    current_quarter += relativedelta(months=3)

pdf_metrics = pd.DataFrame(metrics_data, columns=[
    "PRODUCT_ID", "YEAR_QUARTER", "EU_UNITS_SOLD", "EU_REVENUE_EUR",
    "AVG_SELLING_PRICE_EUR", "EU_MARKET_SHARE_PCT", "CLINICAL_SATISFACTION_SCORE", "NEW_SKU_LAUNCHES"
])

session.create_dataframe(pdf_metrics).write.mode("overwrite").save_as_table("FACT_PRODUCT_METRICS")
print(f"Generated {session.table('FACT_PRODUCT_METRICS').count()} quarterly metrics records")

Generated 300 quarterly metrics records


## Step 7: Generate 250 Clinician Product Reviews

In [ ]:
from datetime import datetime, timedelta
import random

random.seed(42)

products_df = session.table("DIM_PRODUCTS").to_pandas()
categories_df = session.table("DIM_PRODUCT_CATEGORIES").to_pandas()
bridge_df = session.table("BRIDGE_PRODUCT_CATEGORIES").to_pandas()

reviewer_names = [
    "NurseKathryn", "WundExperte", "InfirmiereSophie", "SjukskoterskaAnna", "VerpleegkundigeLisa",
    "TVNurseJames", "WoundCareRN", "SurgeonDrPatel", "ProcurementHans", "CommunityNurseEmma",
    "KliniskSpec", "WundSchwester", "InfirmiereClaire", "NurseOliver", "DrMartinez",
    "TVSpecialist", "WoundNurseKim", "SurgTeamLead", "BuyerMueller", "DistrictNurse",
    "StomaNurseEva", "BurnUnitRN", "DiabeticFootDoc", "PressureUlcerSpec", "PalliativeCareRN",
]

reviewer_countries = [
    "UK", "UK", "UK", "Germany", "Germany",
    "Germany", "France", "France", "Sweden", "Sweden",
    "Netherlands", "Netherlands", "Spain", "Italy", "Denmark",
]

reviewer_roles = ["Wound Care Nurse", "Tissue Viability Specialist", "Surgeon", "Procurement Officer", "Community Nurse"]
role_weights = [35, 20, 15, 15, 15]

care_settings = ["Hospital Acute", "Hospital Chronic", "Community/Home Care", "OR/Surgical"]
care_weights = [40, 25, 20, 15]

wound_types = ["Surgical wound", "Pressure ulcer", "Leg ulcer", "Diabetic foot ulcer", "Burns", "Trauma"]
wound_weights = [25, 20, 20, 15, 10, 10]

start_date = datetime(2023, 1, 1)
end_date = datetime(2026, 2, 28)
date_range = (end_date - start_date).days

base_reviews_per_product = {}
for _, product in products_df.iterrows():
    product_id = int(product["PRODUCT_ID"])
    if product["PRODUCT_NAME"] in ["ALLEVYN", "Aquacel", "Mepilex", "Tegaderm"]:
        base_reviews_per_product[product_id] = 18
    elif product["PRODUCT_NAME"] in ["Biatain", "Aquacel Foam", "Biogel"]:
        base_reviews_per_product[product_id] = 15
    elif product["IS_MOLNLYCKE_PRODUCT"]:
        base_reviews_per_product[product_id] = 10
    else:
        base_reviews_per_product[product_id] = 8

total_reviews = sum(base_reviews_per_product.values())
scale_factor = 275 / total_reviews
for product_id in base_reviews_per_product:
    base_reviews_per_product[product_id] = max(4, int(base_reviews_per_product[product_id] * scale_factor))

def get_rating_for_product(product_name, is_molnlycke, product_type):
    if product_name in ["ALLEVYN", "PICO"]:
        weights = [2, 5, 12, 38, 43]
    elif product_name in ["Aquacel", "Biatain", "Aquacel Foam"]:
        weights = [3, 6, 15, 36, 40]
    elif product_name in ["Mepilex", "Biogel"]:
        weights = [4, 8, 20, 35, 33]
    elif is_molnlycke:
        weights = [5, 10, 22, 35, 28]
    else:
        weights = [4, 8, 18, 35, 35]
    return random.choices([1, 2, 3, 4, 5], weights=weights)[0]

cat_lookup = dict(zip(categories_df["CATEGORY_ID"], categories_df["CATEGORY_NAME"]))

reviews_metadata = []
for _, product in products_df.iterrows():
    product_id = int(product["PRODUCT_ID"])
    product_name = product["PRODUCT_NAME"]
    manufacturer = product["MANUFACTURER"]
    product_type = product["PRODUCT_TYPE"]
    price_tier = product["PRICE_TIER"]
    is_molnlycke = product["IS_MOLNLYCKE_PRODUCT"]

    product_cats = bridge_df[bridge_df["PRODUCT_ID"] == product_id]["CATEGORY_ID"].tolist()
    num_reviews = base_reviews_per_product.get(product_id, 6)

    for _ in range(num_reviews):
        rating = get_rating_for_product(product_name, is_molnlycke, product_type)
        cat_id = random.choice(product_cats) if product_cats else 1
        product_category = cat_lookup.get(cat_id, product["PRIMARY_CATEGORY"])
        reviewer_name = random.choice(reviewer_names) + str(random.randint(1, 999))
        reviewer_country = random.choice(reviewer_countries)
        reviewer_role = random.choices(reviewer_roles, weights=role_weights)[0]
        review_date = (start_date + timedelta(days=random.randint(0, date_range))).strftime("%Y-%m-%d")
        care_setting = random.choices(care_settings, weights=care_weights)[0]
        wound_type = random.choices(wound_types, weights=wound_weights)[0]

        reviews_metadata.append((
            product_id, product_name, manufacturer, product_type, price_tier, is_molnlycke,
            product_category, reviewer_name, reviewer_country, reviewer_role, rating,
            review_date, care_setting, wound_type
        ))

random.shuffle(reviews_metadata)
reviews_metadata = reviews_metadata[:250]

pdf_meta = pd.DataFrame(reviews_metadata, columns=[
    "PRODUCT_ID", "PRODUCT_NAME", "MANUFACTURER", "PRODUCT_TYPE", "PRICE_TIER", "IS_MOLNLYCKE_PRODUCT",
    "PRODUCT_CATEGORY", "REVIEWER_NAME", "REVIEWER_COUNTRY", "REVIEWER_ROLE", "RATING",
    "REVIEW_DATE", "CARE_SETTING", "WOUND_TYPE"
])
session.create_dataframe(pdf_meta).write.mode("overwrite").save_as_table("REVIEWS_STAGING")
print(f"Created staging table with {len(reviews_metadata)} review metadata records")

Created staging table with 250 review metadata records


In [10]:
session.sql("""
CREATE OR REPLACE TABLE REVIEWS AS
SELECT 
    ROW_NUMBER() OVER (ORDER BY REVIEW_DATE) AS REVIEW_ID,
    PRODUCT_ID,
    PRODUCT_CATEGORY,
    REVIEWER_NAME,
    REVIEWER_COUNTRY,
    REVIEWER_ROLE,
    RATING,
    REVIEW_DATE::DATE AS REVIEW_DATE,
    SNOWFLAKE.CORTEX.COMPLETE(
        'claude-sonnet-4-5',
        'Generate a short product review title (5-10 words) for a ' || RATING || '-star review of '
        || PRODUCT_NAME || ' (' || PRODUCT_TYPE || ') by ' || MANUFACTURER || '. '
        || 'Category: ' || PRODUCT_CATEGORY || '. Reviewer role: ' || REVIEWER_ROLE || '. '
        || CASE 
            WHEN RATING >= 4 THEN 'The clinician was satisfied.'
            WHEN RATING = 3 THEN 'The clinician had mixed feelings.'
            ELSE 'The clinician was disappointed.'
           END
        || ' Return ONLY the title, no quotes or explanation.'
    ) AS REVIEW_TITLE,
    SNOWFLAKE.CORTEX.COMPLETE(
        'claude-sonnet-4-5',
        'Write a realistic clinical product review (80-120 words) for a ' || RATING || '-star review of '
        || PRODUCT_NAME || ' (' || PRODUCT_TYPE || ') by ' || MANUFACTURER || '. '
        || 'Category: ' || PRODUCT_CATEGORY || '. Reviewer: ' || REVIEWER_ROLE || ' in ' || REVIEWER_COUNTRY || '. '
        || 'Care setting: ' || CARE_SETTING || '. Wound type: ' || WOUND_TYPE || '. '
        || CASE 
            WHEN IS_MOLNLYCKE_PRODUCT THEN 'This is a Mölnlycke product. Common themes: '
                || CASE WHEN PRODUCT_CATEGORY = 'Foam Dressings' THEN 'Safetac silicone adhesion comfort, gentle removal, absorbency capacity, conformability to wound bed, cost per dressing change.'
                       WHEN PRODUCT_CATEGORY = 'NPWT' THEN 'portability, ease of canister changes, battery life, seal integrity, noise level vs hospital systems.'
                       WHEN PRODUCT_CATEGORY = 'Surgical Gloves' THEN 'tactile sensitivity, puncture indication system, fit comfort during long procedures, latex-free options.'
                       WHEN PRODUCT_CATEGORY IN ('Antimicrobial Dressings', 'Alginate/Fiber Dressings') THEN 'gelling properties, silver efficacy, exudate management, wound bed conformability.'
                       ELSE 'product quality, ease of use, clinical evidence, value for money.' END
            ELSE 'This is a competitor product. '
                || CASE WHEN PRODUCT_NAME IN ('ALLEVYN', 'PICO') THEN 'Known for consistent absorption, strong clinical evidence base, established KOL network, reliable performance.'
                       WHEN PRODUCT_NAME IN ('Aquacel', 'Aquacel Foam', 'Aquacel Ag+') THEN 'Hydrofiber gelling technology, vertical wicking, effective exudate management, well-researched.'
                       WHEN PRODUCT_NAME IN ('Biatain', 'Biatain Ag') THEN 'Good foam structure, reliable adhesion, cost-effective, strong Nordic heritage.'
                       WHEN PRODUCT_NAME IN ('Tegaderm') THEN 'Ubiquitous film dressing, transparent monitoring, strong brand recognition, widely stocked.'
                       ELSE 'Established product with clinical following.' END
           END
        || CASE 
            WHEN RATING = 5 THEN ' Write an enthusiastic review praising specific clinical features (absorption, adhesion, conformability, ease of use, patient comfort).'
            WHEN RATING = 4 THEN ' Write a positive review with one specific clinical concern (cost, availability, sizing options, training needed).'
            WHEN RATING = 3 THEN ' Write a mixed review mentioning both clinical positives and negatives about the product.'
            WHEN RATING = 2 THEN ' Write a negative review with specific product complaints (poor adhesion, maceration, difficult removal, inadequate absorption).'
            ELSE ' Write a very negative review detailing multiple product problems and poor clinical outcomes.'
           END
        || ' Use clinical wound care terminology. Be specific about product features. Return ONLY the review text.'
    ) AS REVIEW_TEXT,
    CARE_SETTING,
    WOUND_TYPE
FROM REVIEWS_STAGING
""").collect()

print(f"Generated {session.table('REVIEWS').count()} AI-written reviews")

SnowparkSQLException: (1304): 01c32718-0108-277a-0041-b7870dcedd72: 000904 (42000): SQL compilation error: error line 14 at position 62
invalid identifier 'MANUFACTURER'
There are existing quoted column identifiers: ['"status"']. Please use one of them to reference the column. See more details on Snowflake identifier requirements https://docs.snowflake.com/en/sql-reference/identifiers-syntax

In [ ]:
session.sql("DROP TABLE IF EXISTS REVIEWS_STAGING").collect()
print("Cleaned up staging table.")

## Step 8: Validate the Data

In [ ]:
print("=== Data Validation Summary ===\n")

print("Row Counts:")
for table in ["DIM_PRODUCTS", "DIM_PRODUCT_CATEGORIES", "BRIDGE_PRODUCT_CATEGORIES", "FACT_PRODUCT_METRICS", "REVIEWS"]:
    count = session.table(table).count()
    print(f"  {table}: {count}")

print("\n--- Rating Distribution ---")
session.sql("""
    SELECT 
        RATING,
        COUNT(*) AS COUNT,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS PERCENTAGE
    FROM REVIEWS
    GROUP BY RATING
    ORDER BY RATING DESC
""").show()

print("\n--- Average Ratings: Mölnlycke vs Competitors ---")
session.sql("""
    SELECT 
        CASE WHEN p.IS_MOLNLYCKE_PRODUCT THEN 'Mölnlycke Products' ELSE 'Competitors' END AS OWNERSHIP,
        ROUND(AVG(r.RATING), 2) AS AVG_RATING,
        COUNT(*) AS REVIEW_COUNT
    FROM REVIEWS r
    JOIN DIM_PRODUCTS p ON r.PRODUCT_ID = p.PRODUCT_ID
    GROUP BY OWNERSHIP
    ORDER BY OWNERSHIP
""").show()

print("\n--- Top 5 Products by Rating (min 5 reviews) ---")
session.sql("""
    SELECT 
        p.PRODUCT_NAME,
        p.MANUFACTURER,
        p.IS_MOLNLYCKE_PRODUCT,
        ROUND(AVG(r.RATING), 2) AS AVG_RATING,
        COUNT(*) AS REVIEW_COUNT
    FROM REVIEWS r
    JOIN DIM_PRODUCTS p ON r.PRODUCT_ID = p.PRODUCT_ID
    GROUP BY p.PRODUCT_ID, p.PRODUCT_NAME, p.MANUFACTURER, p.IS_MOLNLYCKE_PRODUCT
    HAVING COUNT(*) >= 5
    ORDER BY AVG_RATING DESC
    LIMIT 5
""").show()

print("\n--- Quarterly Metrics Sample (Mölnlycke Products, 2025) ---")
session.sql("""
    SELECT 
        p.PRODUCT_NAME,
        m.YEAR_QUARTER,
        m.EU_UNITS_SOLD,
        m.EU_REVENUE_EUR,
        m.EU_MARKET_SHARE_PCT
    FROM FACT_PRODUCT_METRICS m
    JOIN DIM_PRODUCTS p ON m.PRODUCT_ID = p.PRODUCT_ID
    WHERE p.IS_MOLNLYCKE_PRODUCT = TRUE AND m.YEAR_QUARTER >= '2025-01-01'
    ORDER BY p.PRODUCT_NAME, m.YEAR_QUARTER
    LIMIT 6
""").show()

print("\n--- Sample Reviews ---")
session.sql("""
    SELECT 
        p.PRODUCT_NAME,
        r.PRODUCT_CATEGORY,
        r.REVIEWER_ROLE,
        r.RATING,
        r.REVIEW_TITLE,
        LEFT(r.REVIEW_TEXT, 100) || '...' AS REVIEW_PREVIEW
    FROM REVIEWS r
    JOIN DIM_PRODUCTS p ON r.PRODUCT_ID = p.PRODUCT_ID
    ORDER BY RANDOM()
    LIMIT 3
""").show()

print("\n=== Validation Complete ===")